# mijn.ista.nl authentication playground

Interactive lab for the login protocol in `mijn_ista_client`, also used by the Home Assistant integration's published API.

**What you can do here**

1. **Step-by-step login (debugging aid):** walk through the OIDC/Keycloak login one HTTP step at a time and inspect
   status codes, redirects, cookies and Keycloak error messages.
2. **Authentication (final version):** log in with `ResilientMijnIstaAPI`, the authenticator proposed for the integration,
   and test it with an injected fault.
3. **Energy values:** fetch properties, services, the annual comparison, building averages, monthly consumption and
   billing-period meter totals with the same API calls the integration uses.
4. **Session lifetime probe:** check how long a session stays valid, to decide whether a re-login on every refresh is needed.

**Setup**

* Run `uv sync` in the repo root and select the `.venv` kernel.
* Create a `.env` file in the repo root (already git-ignored):
  ```
  ISTA_USERNAME=you@example.com
  ISTA_PASSWORD=...
  ```

**Rate-limit policy** (same as REQUIREMENTS.md NFR-5)

ista publishes no rate limits for mijn.ista.nl / login.ista.com; the values below are a deliberately conservative,
self-imposed policy based on observed behaviour.

* Consumption data is not time-critical: the integration only needs fresh values every **6 h** (`REFRESH_INTERVAL_H`).
* **Logins** — `login.ista.com` answers **HTTP 403** after a few logins in quick succession, so a login (and every
  login retry) starts at least **120 s** (`MIN_LOGIN_GAP_S`) after the previous one — across all sessions in this kernel.
* **Data requests** — calls to `mijn.ista.nl/api/*` (UserValues, MonthValues shard polls, averages, …) are spaced at
  least **5 s** apart (`MIN_REQUEST_GAP_S`). The login's own redirect chain is not throttled.
* Logins go through `LOGIN_GATE`, which prints `⏳` while it waits. Data requests go through `REQUEST_THROTTLE`, which
  every session from `new_session()` applies automatically; it waits silently and counts the total time waited.
  Running all sections top to bottom takes roughly **10 minutes**: three logins (sections 1, 2 and the fault
  injection retry) 120 s apart, plus the data requests.

> ⚠️ Outputs can contain account details. Clear all outputs before committing this notebook.

## 0. Setup

In [ ]:
import asyncio
import html
import logging
import os
import re
import time
from dataclasses import dataclass
from typing import Any

import aiohttp
from yarl import URL
from dotenv import find_dotenv, load_dotenv

from mijn_ista_client import MijnIstaAPI, MijnIstaAuthError, MijnIstaConnectionError
from mijn_ista_client import api as ista_client

load_dotenv(find_dotenv(usecwd=True))
USERNAME = os.environ.get("ISTA_USERNAME", "")
PASSWORD = os.environ.get("ISTA_PASSWORD", "")
LANG = "nl-NL"  # or "en-GB"

BASE_URL = ista_client.BASE_URL
TIMEOUT = aiohttp.ClientTimeout(total=30)

# Rate-limit policy (REQUIREMENTS.md NFR-5). ista publishes no limits; these are self-imposed.
REFRESH_INTERVAL_H = 6  # consumption data only needs refreshing every 6 hours
MIN_LOGIN_GAP_S = 120  # minimum time between login attempts, including retries
MIN_REQUEST_GAP_S = 5  # minimum time between data requests to mijn.ista.nl/api/*

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-7s %(name)s: %(message)s", force=True)
# Set to logging.DEBUG to see the library's own debug output.
logging.getLogger("mijn_ista_client").setLevel(logging.INFO)

if not (USERNAME and PASSWORD):
    print("⚠️  ISTA_USERNAME / ISTA_PASSWORD not set — logins will fail with invalid credentials.")
else:
    print(f"Using account {USERNAME[:3]}***")

### Helpers

* `new_session()` creates an `aiohttp` session with its **own cookie jar** and a request tracer.
* Every request (including followed redirects) is recorded in the session's trace; query strings are stripped because they carry OIDC `state`/`code` values.
* `show_cookies()` lists cookie names/domains only — never values.
* `kc_feedback()` extracts the messages Keycloak renders on its login page (invalid credentials, *cookie not found*, *login timed out*, …).
* `LOGIN_GATE.wait()` enforces `MIN_LOGIN_GAP_S` between login attempts.
* `REQUEST_THROTTLE` enforces `MIN_REQUEST_GAP_S` between `/api/*` data requests; sessions from `new_session()` apply it
  automatically (also to the library's internal shard polling and 401 retries).

In [ ]:
_SESSIONS: list[aiohttp.ClientSession] = []
TRACES: dict[aiohttp.ClientSession, list[tuple[float, str, str, int]]] = {}
LABELS: dict[aiohttp.ClientSession, str] = {}
# Fault injection rules per session: list of [url_substring, status, remaining_count]
FAULTS: dict[aiohttp.ClientSession, list[list]] = {}


def redact(url) -> str:
    return str(url).split("?", 1)[0]


def _fault_middleware(get_session):
    async def middleware(req: aiohttp.ClientRequest, handler):
        for rule in FAULTS.get(get_session(), []):
            needle, status, remaining = rule
            if remaining > 0 and needle in str(req.url):
                rule[2] -= 1
                raise aiohttp.ClientResponseError(
                    req.request_info, (), status=status, message=f"injected {status} on {needle}"
                )
        return await handler(req)

    return middleware


def _throttle_middleware():
    async def middleware(req: aiohttp.ClientRequest, handler):
        if req.url.host == "mijn.ista.nl" and req.url.path.startswith("/api/"):
            await REQUEST_THROTTLE.wait(f"{req.method} {req.url.path}")
        return await handler(req)

    return middleware


class IstaCookieJar(aiohttp.CookieJar):
    """Cookie jar without aiohttp's per-request Cookie-header limit (aiohttp ≥ 3.14.4).

    mijn.ista.nl's ASP.NET auth cookie is split in chunks (`.AspNetCore.Cookies` = "chunks-3" plus C1..C3, ~8.8 kB).
    That exceeds aiohttp's 8190-byte Cookie-header budget, so the stock jar silently drops the largest chunk (C1).
    ASP.NET then can't reassemble the ticket and `/Home` returns 401.
    """

    _limits_enabled = False  # private aiohttp switch, also used by DummyCookieJar


def new_session(label: str = "session") -> aiohttp.ClientSession:
    """Session with a private cookie jar, a request trace and fault-injection hook."""
    trace: list[tuple[float, str, str, int]] = []
    t0 = time.monotonic()

    async def on_redirect(_s, _ctx, p):
        trace.append((time.monotonic() - t0, p.method, redact(p.url), p.response.status))

    async def on_end(_s, _ctx, p):
        status = p.response.status
        # For un-followed redirects, record where the server wanted to send us.
        target = redact(p.response.headers.get("Location", "")) if 300 <= status < 400 else ""
        url = redact(p.url) + (f"  → {target}" if target else "")
        trace.append((time.monotonic() - t0, p.method, url, status))

    async def on_exc(_s, _ctx, p):
        trace.append((time.monotonic() - t0, p.method, redact(p.url), -1))

    tc = aiohttp.TraceConfig()
    tc.on_request_redirect.append(on_redirect)
    tc.on_request_end.append(on_end)
    tc.on_request_exception.append(on_exc)

    holder: list[aiohttp.ClientSession] = []
    session = aiohttp.ClientSession(
        cookie_jar=IstaCookieJar(),
        trace_configs=[tc],
        middlewares=(_throttle_middleware(), _fault_middleware(lambda: holder[0])),
    )
    holder.append(session)
    TRACES[session], LABELS[session] = trace, label
    _SESSIONS.append(session)
    return session


def show_trace(session, since: int = 0) -> None:
    for t, method, url, status in TRACES[session][since:]:
        flag = "✅" if 200 <= status < 400 else "❌"
        shown = "ERR" if status < 0 else status
        print(f"  {t:6.2f}s {flag} {method:4} {shown:>4}  {url}")


def show_cookies(session) -> None:
    cookies = sorted(session.cookie_jar, key=lambda c: (c["domain"], c.key))
    print(f"  {len(cookies)} cookies in jar of '{LABELS[session]}':")
    for c in cookies:
        print(f"    {c['domain'] or '(host-only)':22} {c.key:40} len={len(c.value)}")


def extract(pattern: str, text: str, flags: int = re.S) -> str | None:
    m = re.search(pattern, text, flags)
    return html.unescape(m.group(1)).strip() if m else None


def kc_feedback(page: str) -> dict[str, str]:
    """Messages Keycloak shows on the login page, keyed by where they appear."""
    found = {
        "input-error": extract(r'<span[^>]*id="input-error[^"]*"[^>]*>(.*?)</span>', page),
        "feedback": extract(r'<span[^>]*class="[^"]*kc-feedback-text[^"]*"[^>]*>(.*?)</span>', page),
        "alert": extract(r'<div[^>]*class="[^"]*alert-error[^"]*"[^>]*>(.*?)</div>', page),
        "title": extract(r"<title>(.*?)</title>", page),
    }
    return {k: re.sub(r"<[^>]+>|\s+", " ", v).strip() for k, v in found.items() if v}


def mask(value: Any, keep: int = 4) -> str:
    s = str(value)
    return s[:keep] + "…" if len(s) > keep else s


class Throttle:
    """Kernel-wide throttle: calls to `wait()` return at least `min_gap_s` seconds apart."""

    def __init__(self, min_gap_s: float, what: str, verbose: bool = True) -> None:
        self.min_gap_s = min_gap_s
        self.what = what
        self.verbose = verbose
        self.calls = 0
        self.waited_s = 0.0
        self._last: float | None = None
        self._lock = asyncio.Lock()

    def seconds_until_free(self) -> float:
        if self._last is None:
            return 0.0
        return max(0.0, self._last + self.min_gap_s - time.monotonic())

    async def wait(self, label: str = "login") -> None:
        async with self._lock:
            if (remaining := self.seconds_until_free()) > 0:
                if self.verbose:
                    print(f"  ⏳ {label}: waiting {remaining:.1f}s (min {self.min_gap_s:.0f}s between {self.what})")
                await asyncio.sleep(remaining)
                self.waited_s += remaining
            self.calls += 1
            self._last = time.monotonic()

    def __repr__(self) -> str:
        return f"<Throttle {self.what}: min gap {self.min_gap_s:.0f}s, {self.calls} calls, waited {self.waited_s:.0f}s>"


# Keep the last-call timestamps when this cell is re-run, so the gaps are still respected.
_prev_login, _prev_request = globals().get("LOGIN_GATE"), globals().get("REQUEST_THROTTLE")
LOGIN_GATE = Throttle(MIN_LOGIN_GAP_S, "logins")
REQUEST_THROTTLE = Throttle(MIN_REQUEST_GAP_S, "data requests", verbose=False)
LOGIN_GATE._last = getattr(_prev_login, "_last", None)
REQUEST_THROTTLE._last = getattr(_prev_request, "_last", None)

## 1. Step-by-step login

This mirrors `MijnIstaAPI.authenticate()` exactly, but each HTTP step is its own cell so you can inspect, modify, or repeat it.
Re-run the first cell of this section to start over with an empty jar (it waits for `LOGIN_GATE`).

```
GET  mijn.ista.nl/              → 200, load-balancer affinity cookie
GET  mijn.ista.nl/home/login    → 302 to Keycloak authorize URL (PKCE/state kept server-side)
GET  login.ista.com/…/auth      → 200, login form (kc-form-login)
POST login.ista.com/…/authenticate (username/password) → 200, auto-submit form with `code`
POST mijn.ista.nl/signin-oidc   → 302, ASP.NET session cookie set
GET  mijn.ista.nl/Home          → 200, <meta name="csrf-token">
POST mijn.ista.nl/api/…         → data, with CSRF header + placeholder bearer
```

> **Cookie-chunk pitfall.** The ASP.NET session cookie is ~8.8 kB and split into `.AspNetCore.Cookies` + `C1..C3`.
> Since aiohttp 3.14.4 the stock `CookieJar` caps the `Cookie` header at 8190 bytes and silently drops the largest chunk
> (`C1`), so `/Home` returns 401 and no CSRF token is found. Home Assistant 2026.10 ships aiohttp 3.14.4. All sessions here
> therefore use `IstaCookieJar` (limits disabled); step 6 shows which chunks are sent.

In [ ]:
await LOGIN_GATE.wait("manual login")
s = new_session("manual")
state: dict[str, Any] = {}

# Step 1 — priming request (sets load-balancer affinity cookie)
async with s.get(f"{BASE_URL}/", timeout=TIMEOUT) as resp:
    print("final:", resp.status, redact(resp.url))
show_trace(s)
show_cookies(s)

In [ ]:
# Step 2 — start the OIDC challenge; expect 302 with a Keycloak authorize URL
async with s.get(
    f"{BASE_URL}/home/login",
    headers={
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer": f"{BASE_URL}/",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Site": "same-origin",
    },
    timeout=TIMEOUT,
    allow_redirects=False,
) as resp:
    state["authorize_url"] = resp.headers.get("Location") if resp.status == 302 else None
    print("status:", resp.status, "| Location:", redact(resp.headers.get("Location", "-")))

In [ ]:
# Step 3 — load the Keycloak login page and find the form action
if not state.get("authorize_url"):
    print("⏭  skipped: no authorize URL from step 2")
else:
    async with s.get(state["authorize_url"], timeout=TIMEOUT) as resp:
        login_html = await resp.text()
        print("status:", resp.status, redact(resp.url))
    state["form_action"] = extract(r'<form[^>]+id="kc-form-login"[^>]+action="([^"]+)"', login_html)
    print("form action:", redact(state["form_action"]) if state["form_action"] else "❌ NOT FOUND")
    print("keycloak feedback:", kc_feedback(login_html) or "-")
    show_cookies(s)

In [ ]:
# Step 4 — submit credentials; expect an auto-submit form carrying `code`
if not state.get("form_action"):
    print("⏭  skipped: no login form from step 3")
else:
    async with s.post(
        state["form_action"],
        data={"username": USERNAME, "password": PASSWORD, "credentialId": ""},
        timeout=TIMEOUT,
        allow_redirects=False,
    ) as resp:
        body = await resp.text()
        print("status:", resp.status)
    state["callback_action"] = extract(r'ACTION="([^"]+)"', body, 0)
    state["callback_fields"] = {k: html.unescape(v) for k, v in re.findall(r'NAME="([^"]+)" VALUE="([^"]*)"', body)}
    if 'id="kc-form-login"' in body:
        # The library maps this to MijnIstaAuthError("Invalid credentials") — but Keycloak
        # also re-renders the form for e.g. "cookie not found" or "login timed out".
        print("❌ login form re-rendered. Keycloak says:", kc_feedback(body) or "(no message)")
    elif state["callback_action"] and "code" in state["callback_fields"]:
        print("✅ got authorization code, callback →", redact(state["callback_action"]))
        print("   fields:", sorted(state["callback_fields"]))
    else:
        print("❌ unexpected response:", kc_feedback(body) or body[:300])

In [ ]:
# Step 5 — hand the code to mijn.ista.nl; expect 302/303 and a session cookie
if "code" not in state.get("callback_fields", {}):
    print("⏭  skipped: no authorization code from step 4")
else:
    async with s.post(
        state["callback_action"], data=state["callback_fields"], timeout=TIMEOUT, allow_redirects=False
    ) as resp:
        print("status:", resp.status, "| Location:", redact(resp.headers.get("Location", "-")))
        state["session_ok"] = resp.status in (302, 303)
    show_cookies(s)

In [ ]:
# Step 6 — fetch the CSRF token from the dashboard
# Non-200 here is mapped by the library to MijnIstaAuthError → HA reauth, even if it was transient!
if not state.get("session_ok"):
    print("⏭  skipped: session callback did not succeed")
else:
    async with s.get(f"{BASE_URL}/Home", timeout=TIMEOUT) as resp:
        page = await resp.text()
        print("status:", resp.status, redact(resp.url), "| redirects:", [r.status for r in resp.history])
    state["csrf"] = extract(r'<meta name="csrf-token" content="([^"]+)"', page)
    print("csrf token:", mask(state["csrf"], 6) if state["csrf"] else "❌ NOT FOUND")

    # All auth-cookie chunks must be sent, otherwise /Home answers 401 (see IstaCookieJar).
    url = URL(f"{BASE_URL}/Home")
    in_jar = sorted(c.key for c in s.cookie_jar if c.key.startswith(".AspNetCore.Cookies"))
    sent = set(s.cookie_jar.filter_cookies(url))
    stock = aiohttp.CookieJar()
    stock.update_cookies({c.key: c for c in s.cookie_jar}, url)
    stock_sent = set(stock.filter_cookies(url))
    for name in in_jar:
        print(f"  {name:24} sent: {'✅' if name in sent else '❌'}   (stock aiohttp jar: {'✅' if name in stock_sent else '❌'})")

### Get `/api/Values/UserValues`

Run these data cells in order after step 6. They reuse the manually authenticated session through `MijnIstaAPI`.
`UserValues` only needs `LANG`; the other endpoints also need the property's `Cuid`. Meter totals require a
`Billingperiod`, and averages require `PAR` with start/end dates. The client supplies those payloads and polls
monthly shards (including HTTP 425 responses). Billing request failures raise errors; missing monthly data is
reported explicitly instead of silently becoming `None`.

The three property-specific results below are dictionaries keyed by `Cuid`, so accounts with multiple properties
keep every response. Billing requests use the newest available billing period, whose label can be a future year.

In [ ]:
# Step 7 — reuse the cookie session and CSRF token from the manual login.
if not state.get("csrf"):
    raise RuntimeError("No CSRF token: run the manual login through step 6 first")

manual_api = MijnIstaAPI(s, USERNAME, PASSWORD, lang=LANG)
manual_api._csrf_token = state["csrf"]
user_values = await manual_api.get_user_values()
manual_customers = user_values.get("Cus") or []
manual_periods = {}
print("display name:", mask(user_values.get("DisplayName"), 3))
if not manual_customers:
    print("No properties returned by UserValues")
for cus in manual_customers:
    cur = cus.get("curConsumption") or {}
    services = [b.get("Description") for b in cur.get("Billingservices") or []]
    manual_periods[cus["Cuid"]] = max(cur.get("BillingPeriods") or [], key=lambda p: p["y"], default=None)
    print(f"  cuid={mask(cus['Cuid'], 6)} services={services}")
show_trace(s)

In [ ]:
user_values

### Get `/api/Consumption/MonthValues`

In [ ]:
month_values = {}
for cus in manual_customers:
    cuid = cus["Cuid"]
    month_values[cuid] = await manual_api.get_month_values(cuid)
    print(f"cuid={mask(cuid, 6)}: {len(month_values[cuid].get('mc') or [])} monthly entries")
    if not month_values[cuid].get("mc"):
        print("No monthly consumption returned within the polling budget")
show_trace(s)

In [ ]:
month_values

### Get `/api/Values/ConsumptionValues`

In [ ]:
consumption_values = {}
for cus in manual_customers:
    cuid = cus["Cuid"]
    period = manual_periods[cuid]
    if period is None:
        print(f"cuid={mask(cuid, 6)}: no billing period, meter totals skipped")
        continue
    consumption_values[cuid] = await manual_api.get_consumption_values(cuid, period)
    print(f"cuid={mask(cuid, 6)}: meter totals for billing period {period['y']}")
show_trace(s)

In [ ]:
consumption_values

### Get `/api/Values/ConsumptionAverages`

In [ ]:
consumption_averages = {}
for cus in manual_customers:
    cuid = cus["Cuid"]
    period = manual_periods[cuid]
    if period is None:
        print(f"cuid={mask(cuid, 6)}: no billing period, building averages skipped")
        continue
    start, end = period["s"][:10], period["e"][:10]
    consumption_averages[cuid] = await manual_api.get_consumption_averages(cuid, start, end)
    print(f"cuid={mask(cuid, 6)}: building averages for {start} to {end}")
show_trace(s)

In [ ]:
consumption_averages

## 2. Authentication (final version)

`ResilientMijnIstaAPI` is the authenticator proposed for the integration (`api.py` in cycle 1). It is a drop-in
`MijnIstaAPI` subclass; only `authenticate()` changes:

1. **Private cookie jar.** Use one session per config entry (`async_create_clientsession`) and clear the jar before
   every attempt. Stale ista cookies (the 403 bug, aalaei/mijn-ista-api#1) and other entries therefore can't interfere.
   The jar must be an `IstaCookieJar` (no Cookie-header size limit), otherwise aiohttp ≥ 3.14.4 drops auth-cookie chunk
   `C1` and every login ends in `/Home` 401 → `MijnIstaAuthError`. This was the main cause of the unstable authentication.
2. **Up to 3 attempts, each gated by `LOGIN_GATE`** (≥ `MIN_LOGIN_GAP_S` = 120 s apart), so ista's login rate limiter
   is never triggered by our retries. The worst case is about 4–5 minutes, which doesn't matter against a 6 h refresh interval.
3. **Classify failures before giving up:**

| Outcome | Raised | Home Assistant |
|---|---|---|
| login succeeds on any attempt | — | data refresh continues |
| "Invalid credentials" on **two consecutive** attempts | `MijnIstaAuthError` | `ConfigEntryAuthFailed` → reauth |
| anything else after 3 attempts (403/5xx, timeouts, Keycloak "cookie not found", `/Home` ≠ 200, …) | `MijnIstaConnectionError` | `UpdateFailed` → retry later, no reauth |

The library's `_post()` and `_poll_shard()` call `self.authenticate()` on HTTP 401. That means re-logins in the middle of a
refresh go through this same gated, retrying path too.

In [ ]:
@dataclass
class AttemptResult:
    attempt: int
    ok: bool
    error: str | None
    seconds: float


def _is_invalid_credentials(exc: Exception) -> bool:
    return isinstance(exc, MijnIstaAuthError) and str(exc) == "Invalid credentials"


class ResilientMijnIstaAPI(MijnIstaAPI):
    """Retrying, rate-limited, self-cleaning login on a private cookie jar."""

    def __init__(self, *args, attempts: int = 3, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self._attempts = attempts
        self.history: list[AttemptResult] = []

    async def authenticate(self) -> None:
        last: Exception | None = None
        for attempt in range(1, self._attempts + 1):
            await LOGIN_GATE.wait(f"login attempt {attempt}/{self._attempts}")
            self._session.cookie_jar.clear()  # private jar: start every attempt from scratch
            t = time.monotonic()
            try:
                await super().authenticate()
            except (MijnIstaAuthError, MijnIstaConnectionError) as exc:
                self.history.append(AttemptResult(attempt, False, f"{type(exc).__name__}: {exc}", time.monotonic() - t))
                logging.getLogger("resilient").warning("login attempt %d/%d failed: %s", attempt, self._attempts, exc)
                if _is_invalid_credentials(exc) and last is not None and _is_invalid_credentials(last):
                    raise  # confirmed twice in a row: really bad credentials
                last = exc
                continue
            self.history.append(AttemptResult(attempt, True, None, time.monotonic() - t))
            return
        raise MijnIstaConnectionError(f"login failed after {self._attempts} attempts: {last}") from last


async def login(api: ResilientMijnIstaAPI, label: str = "login") -> bool:
    """Authenticate and report the outcome; on failure show the HTTP trace of this login."""
    session = api._session
    mark = len(TRACES[session])
    t = time.monotonic()
    try:
        await api.authenticate()
    except (MijnIstaAuthError, MijnIstaConnectionError) as exc:
        print(f"❌ {label}: {type(exc).__name__}: {exc}")
        show_trace(session, mark)
        return False
    print(f"✅ {label}: OK in {time.monotonic() - t:.1f}s (incl. waiting for the login gate)")
    return True

In [ ]:
ista_session = new_session("ista")
api = ResilientMijnIstaAPI(ista_session, USERNAME, PASSWORD, lang=LANG)
await login(api)
for r in api.history:
    print(" ", r)
print("\nHTTP trace of the login:")
show_trace(ista_session)
show_cookies(ista_session)

### Fault injection

Simulate a transient failure: make one step of the login fail once and check that the authenticator recovers on
the next attempt. The retry waits for the login gate, so this cell takes at least `MIN_LOGIN_GAP_S`.
It uses its own session, so `api` above stays logged in.

In [ ]:
def inject_fault(session: aiohttp.ClientSession, url_part: str, status: int = 502, times: int = 1) -> None:
    """Make the next `times` requests whose URL contains `url_part` fail with HTTP `status`."""
    FAULTS.setdefault(session, []).append([url_part, status, times])


# Try other steps too, e.g. "/Home" (CSRF page), "login-actions/authenticate" or "/signin-oidc".
FAULT_URL, FAULT_STATUS = "/home/login", 502

flaky_session = new_session("flaky")
flaky = ResilientMijnIstaAPI(flaky_session, USERNAME, PASSWORD, lang=LANG)
inject_fault(flaky_session, FAULT_URL, FAULT_STATUS)
print(f"— one injected HTTP {FAULT_STATUS} on {FAULT_URL} —")
await login(flaky, "flaky login")
for r in flaky.history:
    print(" ", r)
await flaky_session.close()

## 3. Energy values

The same calls the coordinator makes on every refresh, on the session that is logged in above. Each data request
waits for `REQUEST_THROTTLE` (≥ `MIN_REQUEST_GAP_S` = 5 s apart). If the session has expired, the library logs in again
through `ResilientMijnIstaAPI.authenticate()`.

| Endpoint | Method | Used for |
|---|---|---|
| `/api/Values/UserValues` | `get_user_values()` | properties (Cuid), services and units, billing periods, annual comparison |
| `/api/Values/ConsumptionAverages` | `get_consumption_averages()` | building averages per service for a period |
| `/api/Consumption/MonthValues` | `get_month_values()` | monthly consumption per service and meter (polled in shards) |
| `/api/Values/ConsumptionValues` | `get_consumption_values()` | meter totals for one billing period (not used by the integration yet) |

Raw responses are kept in `RAW` for your own exploration; `peek(obj)` prints their structure.

In [ ]:
RAW: dict[str, Any] = {}


def peek(obj: Any, depth: int = 3, indent: str = "", key: str = "") -> None:
    """Print the structure of a JSON value: keys, types, list lengths and short scalar samples."""
    label = f"{key}: " if key else ""
    if isinstance(obj, dict):
        print(f"{indent}{label}{{{len(obj)} keys}}")
        if depth > 0:
            for k, v in obj.items():
                peek(v, depth - 1, indent + "  ", str(k))
    elif isinstance(obj, list):
        print(f"{indent}{label}[{len(obj)} items]")
        if obj and depth > 0:
            peek(obj[0], depth - 1, indent + "  ", "[0]")
    else:
        print(f"{indent}{label}{type(obj).__name__} = {str(obj)[:40]!r}")


def num(value: Any, dec: int = 2) -> str:
    return f"{value:,.{dec}f}" if isinstance(value, (int, float)) else "-"


def current_period(periods: list[dict[str, Any]]) -> dict[str, Any] | None:
    """Billing period of the current year, else the most recent one (same rule as the coordinator)."""
    year = time.localtime().tm_year
    newest_first = sorted(periods, key=lambda p: p.get("y", 0), reverse=True)
    return next((p for p in newest_first if p.get("y") == year), newest_first[0] if newest_first else None)


RAW["user_values"] = user_values = await api.get_user_values()
customers = user_values.get("Cus", [])
SERVICES: dict[str, dict[int, dict[str, Any]]] = {}  # cuid → service id → Billingservice

print(f"{len(customers)} propert{'y' if len(customers) == 1 else 'ies'}")
for cus in customers:
    cuid, cur = cus["Cuid"], cus.get("curConsumption", {})
    SERVICES[cuid] = {svc["Id"]: svc for svc in cur.get("Billingservices", [])}
    print(f"\n🏠 cuid {mask(cuid, 6)} — {mask(cus.get('Adress', ''), 6)}, {cus.get('City', '')} (since {cus.get('DateStart', '')[:10]})")
    for sid, svc in SERVICES[cuid].items():
        print(f"   service {sid}: {svc.get('Description')} [{svc.get('Unit')}] meter type {svc.get('MeterType')}")
    for p in sorted(cur.get("BillingPeriods", []), key=lambda p: p.get("y", 0), reverse=True):
        print(f"   billing period {p.get('y')}: {p.get('s', '')[:10]} → {p.get('e', '')[:10]}, avg outdoor temp {p.get('ta')}")

### Annual comparison (from `UserValues`)

Current billing period so far vs. the same part of the previous period, per service — the data behind the annual sensors.

In [ ]:
for cus in customers:
    cuid = cus["Cuid"]
    print(f"🏠 cuid {mask(cuid, 6)}")
    print(f"   {'service':24} {'now':>12} {'previous':>12} {'diff %':>8} {'whole prev.':>12}  unit")
    for sc in cus.get("curConsumption", {}).get("ServicesComp", []):
        svc = SERVICES[cuid].get(sc["Id"], {})
        dec = sc.get("DecPos", 2)
        print(
            f"   {svc.get('Description', sc['Id'])!s:24} {num(sc.get('TotalNow'), dec):>12} {num(sc.get('TotalPrevious'), dec):>12}"
            f" {num(sc.get('TotalDiffperc'), 1):>8} {num(sc.get('TotalWholePrevious'), dec):>12}  {svc.get('Unit', '')}"
        )
        for m in sc.get("CurMeters", []):
            print(
                f"     meter {m.get('MeterNr')}: {m.get('BsDate', '')[:10]} {num(m.get('BeginValue'), dec)} → "
                f"{m.get('EsDate', '')[:10]} {num(m.get('EndValue'), dec)} = {num(m.get('CValue'), dec)}"
            )

### Building averages (`ConsumptionAverages`)

Normalised averages of the whole building for the current billing period, used for the "building average" attributes.

In [ ]:
RAW["averages"] = {}
for cus in customers:
    cuid = cus["Cuid"]
    period = current_period(cus.get("curConsumption", {}).get("BillingPeriods", []))
    if not period:
        print(f"🏠 cuid {mask(cuid, 6)}: no billing period, skipped")
        continue
    start, end = period["s"][:10], period["e"][:10]
    RAW["averages"][cuid] = avg = await api.get_consumption_averages(cuid, start, end)
    print(f"🏠 cuid {mask(cuid, 6)}  period {start} → {end}")
    for a in avg.get("Averages", []):
        svc = SERVICES[cuid].get(a.get("BillingServiceId"), {})
        print(f"   {svc.get('Description', a.get('BillingServiceId'))!s:24} {num(a.get('NormalizedValue')):>12}  {svc.get('Unit', '')}")

### Monthly consumption (`MonthValues`)

The server builds this response in shards (`hs` loaded of `sh` total); the library polls until all shards are in or its
time budget runs out (15 s with `quick=True`, 60 s otherwise). With the 5 s request gap that is about 3 or 12 polls.
Check below whether the full history still loads (REQUIREMENTS.md NFR-5).

In [ ]:
QUICK = False  # True = what the coordinator does on its first refresh after HA start
MONTHS_SHOWN = 13

RAW["months"] = {}
for cus in customers:
    cuid = cus["Cuid"]
    t = time.monotonic()
    RAW["months"][cuid] = month = await api.get_month_values(cuid, quick=QUICK)
    entries = sorted(month.get("mc", []), key=lambda mc: (mc["y"], mc["m"]), reverse=True)
    if "sh" not in month:
        shards = "no shard info"
    elif month["sh"] > 0 and month.get("hs", 0) >= month["sh"]:
        shards = f"shards {month['hs']}/{month['sh']} ✅ complete"
    else:
        shards = f"shards {month.get('hs', 0)}/{month['sh']} ⚠️ incomplete"
    print(f"🏠 cuid {mask(cuid, 6)}: {len(entries)} months, {shards}, {time.monotonic() - t:.0f}s")
    services = SERVICES[cuid]
    header = "".join(f"{svc.get('Description', sid)!s:>22}" for sid, svc in services.items())
    print(f"   {'month':8} {'temp':>6}{header}")
    for mc in entries[:MONTHS_SHOWN]:
        by_service = {sc["ServiceId"]: sc for sc in mc.get("ServiceConsumptions", [])}
        cells = ""
        for sid, svc in services.items():
            sc = by_service.get(sid)
            if sc:
                approx = "~" if sc.get("HasApproximation") else ""
                value = f"{num(sc.get('TotalConsumption'))}{approx} {svc.get('Unit', '')}"
            else:
                value = "-"
            cells += f"{value:>22}"
        print(f"   {mc['y']}-{mc['m']:02d}  {num(mc.get('at') or None, 1):>6}{cells}")
    print("   ~ = contains approximated values; temp = average outdoor temperature (KNMI), '-' when not yet available")
print(REQUEST_THROTTLE)

In [ ]:
# Per-meter detail for the most recent month that has data
for cuid, month in RAW["months"].items():
    entries = sorted(month.get("mc", []), key=lambda mc: (mc["y"], mc["m"]), reverse=True)
    latest = next((mc for mc in entries if mc.get("ServiceConsumptions")), None)
    if not latest:
        print(f"🏠 cuid {mask(cuid, 6)}: no monthly data")
        continue
    print(f"🏠 cuid {mask(cuid, 6)} — {latest['y']}-{latest['m']:02d}")
    for sc in latest["ServiceConsumptions"]:
        svc = SERVICES[cuid].get(sc["ServiceId"], {})
        print(
            f"   {svc.get('Description', sc['ServiceId'])}: total {num(sc.get('TotalConsumption'))} {svc.get('Unit', '')},"
            f" building avg {num(sc.get('BuldingAverage'))}"
        )
        for d in sc.get("DeviceConsumptions", []):
            print(
                f"     meter {d.get('SerialNr')}: {d.get('SDate', '')[:10]} {num(d.get('SValue'))} → "
                f"{d.get('EDate', '')[:10]} {num(d.get('EValue'))} = {num(d.get('CValue'))}"
            )

### Meter totals per billing period (`ConsumptionValues`)

Not used by the integration yet. This shows the structure of the response for the current billing period, as input for future sensors.

In [ ]:
RAW["consumption_values"] = {}
for cus in customers:
    cuid = cus["Cuid"]
    period = current_period(cus.get("curConsumption", {}).get("BillingPeriods", []))
    if not period:
        continue
    RAW["consumption_values"][cuid] = values = await api.get_consumption_values(cuid, period)
    print(f"🏠 cuid {mask(cuid, 6)}  billing period {period.get('y')}")
    peek(values, depth=3, indent="   ")

## 4. Session lifetime probe (optional, slow)

Does the ASP.NET session survive a full refresh interval (`REFRESH_INTERVAL_H` = 6 h)? If it does, the coordinator can
*try the data call first* and only log in on 401 — most refreshes then need **no login at all**. The probe logs in once
and then only makes data calls (without automatic re-login). Set `RUN_LIFETIME_PROBE = True` to run it.

In [ ]:
REFRESH_INTERVAL_H = 1 # For testing purposes

In [ ]:
RUN_LIFETIME_PROBE = True
PROBE_EVERY_MIN = 30
PROBE_MAX_HOURS = REFRESH_INTERVAL_H + 1  # a bit beyond one refresh interval

if RUN_LIFETIME_PROBE:
    ps = new_session("lifetime")
    probe = ResilientMijnIstaAPI(ps, USERNAME, PASSWORD, lang=LANG)
    if await login(probe, "lifetime probe login"):
        started = time.monotonic()
        while time.monotonic() - started < PROBE_MAX_HOURS * 3600:
            await asyncio.sleep(PROBE_EVERY_MIN * 60)
            elapsed = time.monotonic() - started
            async with ps.post(
                f"{BASE_URL}/api/Values/UserValues",
                json={"LANG": LANG},
                headers=probe._auth_headers(),
                timeout=TIMEOUT,
                allow_redirects=False,
            ) as resp:
                print(f"after {elapsed / 60:6.1f} min: HTTP {resp.status}")
                if resp.status != 200:
                    break
        else:
            print(f"✅ session still valid after {PROBE_MAX_HOURS} h — longer than the {REFRESH_INTERVAL_H} h refresh interval")
else:
    print("skipped (set RUN_LIFETIME_PROBE = True)")

## 5. Cleanup

In [ ]:
for sess in _SESSIONS:
    if not sess.closed:
        await sess.close()
print(f"closed {len(_SESSIONS)} sessions")